In [2]:
import numpy as np
import random
import matplotlib.pyplot as plt
from geopy.distance import geodesic


In [3]:
ciudades = [
{"nombre": "Ciudad de Mexico", "coords": (19.4326, -99.1332)},
{"nombre": "Nueva York", "coords": (40.7128, -74.0060)},
{"nombre": "Londres", "coords": (51.5074, -0.1278)},
{"nombre": "Tokio", "coords": (35.6762, 139.6503)},
{"nombre": "Sidney", "coords": (-33.8688, 151.2093)},
{"nombre": "Paris", "coords": (48.8566, 2.3522)},
{"nombre": "Berlin", "coords": (52.5200, 13.4050)},
{"nombre": "Roma", "coords": (41.9028, 12.4964)},
{"nombre": "Sao Paulo", "coords": (-23.5505, -46.6333)},
{"nombre": "Buenos Aires", "coords": (-34.6037, -58.3816)},
{"nombre": "El Cairo", "coords": (30.0444, 31.2357)},
{"nombre": "Moscu", "coords": (55.7558, 37.6173)},
{"nombre": "Beijing", "coords": (39.9042, 116.4074)},
{"nombre": "Delhi", "coords": (28.6139, 77.2090)},
{"nombre": "Johannesburgo", "coords": (-26.2041, 28.0473)},
{"nombre": "Toronto", "coords": (43.6532, -79.3832)},
{"nombre": "Dubai", "coords": (25.2769, 55.2962)},
{"nombre": "Singapur", "coords": (1.3521, 103.8198)},
{"nombre": "Estocolmo", "coords": (59.3293, 18.0686)},
{"nombre": "Auckland", "coords": (-36.8485, 174.7633)}
]

In [4]:
def calcular_matriz_distancias(ciudades):
    n = len(ciudades)
    matriz = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            dist = geodesic(ciudades[i]["coords"], ciudades[j]["coords"]).km
            matriz[i][j] = matriz[j][i] = dist
    return matriz

In [5]:
def calcular_fitness(individuo, matriz_distancias):
    distancia_total = 0
    for i in range(len(individuo) - 1):
        distancia_total += matriz_distancias[individuo[i]][individuo[i+1]]
    
    distancia_total += matriz_distancias[individuo[-1]][individuo[0]]
    return distancia_total

In [6]:
individuo = [0, 1]
matriz_distancia = calcular_matriz_distancias(ciudades)

In [7]:
calcular_fitness(individuo, matriz_distancia)

np.float64(6715.390400452742)

In [8]:
def seleccion_torneo(poblacion, fitness_pob, k=5):
    # Selecciona los 2 mejores de entre k padres aleatorios [cite: 143, 153]
    seleccionados = random.sample(list(zip(poblacion, fitness_pob)), k)
    seleccionados.sort(key=lambda x: x[1])
    return seleccionados[0][0], seleccionados[1][0]

In [ ]:
def mutacion_intercambio(individuo, tasa_mutacion):
    if random.random() < tasa_mutacion:
        idx1, idx2 = random.sample(range(1, len(individuo)), 2)
        individuo[idx1], individuo[idx2] = individuo[idx2], individuo[idx1]
        
    return individuo

In [ ]:
def mutacion_insercion(individuo, tasa_mutacion):
    if random.random() < tasa_mutacion:
        gen_origen = random.randint(1, len(individuo) - 1)
        gen = individuo.pop(gen_origen)
        
        gen_destino = random.randint(1, len(individuo) - 1)
        individuo.insert(gen_destino, gen)
        
    return individuo

In [ ]:
def mutacion_desordenada(individuo, tasa_mutacion):
    if random.random() < tasa_mutacion:
        size = len(individuo)

        gen_1, gen_2 = sorted(random.sample(range(1, size), 2))

        subsegmento = individuo[gen_1:gen_2+1]
        random.shuffle(subsegmento)
        
        individuo[gen_1:gen_2+1] = subsegmento

    return individuo

In [42]:
def mutacion_inversion(individuo, tasa_mutacion):
    if random.random() < tasa_mutacion:
        size = len(individuo)

        idx1, idx2 = sorted(random.sample(range(1, size), 2))
        
        individuo[idx1:idx2+1] = individuo[idx1:idx2+1][::-1]
        
    return individuo

In [15]:
def cruce_pmx(padre_1, padre_2):
    
    tamanio = len(padre_1)
    
    gen_1, gen_2 = sorted(random.sample(range(1, tamanio), 2))
    
    h1, h2 = [None]*tamanio, [None]*tamanio
    h1[0], h2[0] = 0, 0 
    
    h1[gen_1:gen_2] = padre_1[gen_1:gen_2]
    h2[gen_1:gen_2] = padre_2[gen_1:gen_2]
    
    def rellenar(hijo, padre):
        for i in range(1, tamanio):
            if hijo[i] is None:
                val = padre[i]
                while val in hijo:
                    gen_in_padre = padre.index(val)
                    if hijo[gen_in_padre] is None:
                        break
                    val = hijo[gen_in_padre]
                hijo[i] = val
                
    rellenar(h1, padre_2)
    rellenar(h2, padre_1)
    return h1, h2

In [49]:
def cruce_orden(p1, p2):
    tamanio = len(p1)
    h1, h2 = [None]*tamanio, [None]*tamanio
    
    h1[0], h2[0] = 0, 0
    
    cruce_1, cruce_2 = sorted(random.sample(range(1, tamanio), 2))
    
    h1[cruce_1:cruce_2+1] = p1[cruce_1:cruce_2+1]
    h2[cruce_1:cruce_2+1] = p2[cruce_1:cruce_2+1]
    
    def rellenar_orden(hijo, padre, corte2):
        size = len(padre)
        pos_hijo = (corte2 + 1) % size
        pos_padre = (corte2 + 1) % size
        
        while None in hijo:
            if pos_hijo == 0:
                pos_hijo = 1
                continue
            
            ciudad_candidata = padre[pos_padre % size]
            if ciudad_candidata not in hijo:
                hijo[pos_hijo] = ciudad_candidata
                pos_hijo = (pos_hijo + 1) % size
            pos_padre = (pos_padre + 1) % size

    rellenar_orden(h1, p2, cruce_2)
    rellenar_orden(h2, p1, cruce_2)
    return h1, h2

In [60]:
import random

def cruce_ciclos(p1, p2):
    tamanio = len(p1)
    h1, h2 = [None]*tamanio, [None]*tamanio
    
    h1[0], h2[0] = 0, 0

    def resolver_ciclo(hijo, p_donante, p_otro):
        try:
            inicio = next(i for i, v in enumerate(hijo) if v is None)
        except StopIteration:
            return 
            
        current = inicio
        while hijo[current] is None:
            hijo[current] = p_donante[current]
            valor_buscado = p_otro[current]
            current = list(p_donante).index(valor_buscado)

    turno_p1 = True
    while None in h1:
        if turno_p1:
            resolver_ciclo(h1, p1, p2)
            resolver_ciclo(h2, p2, p1)
        else:
            resolver_ciclo(h1, p2, p1)
            resolver_ciclo(h2, p1, p2)
        turno_p1 = not turno_p1
        
    return h1, h2

In [ ]:
def cruce_bordes(p1, p2):
    size = len(p1)
    
    # 1. Construir tabla de adyacencias
    adj = {i: set() for i in range(size)}
    for p in [p1, p2]:
        for i in range(size):
            # Vecinos circularmente
            v1 = p[(i - 1) % size]
            v2 = p[(i + 1) % size]
            adj[p[i]].add(v1)
            adj[p[i]].add(v2)
    
    def generar_hijo(adj_copy):
        hijo = []
        curr = 0 # Empezamos siempre en CDMX
        
        while len(hijo) < size:
            hijo.append(curr)
            # Eliminar curr de todas las listas de vecinos
            for v in adj_copy:
                adj_copy[v].discard(curr)
            
            if len(hijo) == size: break
            
            vecinos = list(adj_copy[curr])
            if vecinos:
                # Elegir el vecino con menos vecinos restantes
                # (Estrategia de la "corbata" o grado mínimo)
                curr = min(vecinos, key=lambda x: len(adj_copy[x]))
            else:
                # Si no hay vecinos, elegir uno al azar de los que faltan
                restantes = [n for n in range(size) if n not in hijo]
                curr = random.choice(restantes)
        return hijo

    # Crear copias profundas para no dañar la tabla original
    import copy
    h1 = generar_hijo(copy.deepcopy(adj))
    h2 = generar_hijo(copy.deepcopy(adj))
    return h1, h2

In [59]:
p1 = [0, 1, 2, 3, 4, 5, 6, 7, 8]
p2 = [3, 5, 1, 8, 0, 2, 7, 6, 4]
cruce_ciclos(p1, p2)

([0, 1, 2, 8, 4, 5, 7, 6, 8], [0, 5, 1, 3, 4, 2, 7, 6, 8])